# PyPSA-GB — explore in PyPSA

Loaded straight from the [model-verse](https://modelverse.bayesian.energy) marketplace. This notebook downloads the published PyPSA network and takes a quick look at its buses, generators, snapshots and installed capacity.

_Open-source PyPSA community model. Run all cells (Runtime → Run all)._


## Setup


In [ ]:
%pip install -q pypsa netcdf4 matplotlib

In [ ]:
# Config — the published NetCDF for this model
NC_URL = "https://pub-074fa9bfa2c944399f48bcd7c0d531c7.r2.dev/pypsa-gb/pypsa-gb_Historical_2020_zonal.nc"
MODEL_NAME = "PyPSA-GB"

## Download the network

The file is served as gzip-compressed bytes, so we fetch with a browser user-agent and decompress if needed.


In [ ]:
import urllib.request, os, gzip, shutil

nc_path = "model.nc"
if not os.path.exists(nc_path):
    req = urllib.request.Request(NC_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open("model.bin", "wb") as f:
        shutil.copyfileobj(r, f)
    with open("model.bin", "rb") as f:
        is_gzip = f.read(2) == b"\x1f\x8b"
    if is_gzip:
        with gzip.open("model.bin", "rb") as fin, open(nc_path, "wb") as fout:
            shutil.copyfileobj(fin, fout)
    else:
        os.replace("model.bin", nc_path)
print(f"{os.path.getsize(nc_path)/1e6:.1f} MB ready")

## Load with PyPSA


In [ ]:
import pypsa

n = pypsa.Network(nc_path)
n

## What's in the network?


In [ ]:
print(f'Buses:      {len(n.buses)}')
print(f'Generators: {len(n.generators)}')
print(f'Lines:      {len(n.lines)}')
print(f'Links:      {len(n.links)}')
print(f'Snapshots:  {len(n.snapshots)}  ({n.snapshots[0]} → {n.snapshots[-1]})')
print(f'Carriers:   {sorted(n.generators.carrier.unique())}')

In [ ]:
n.buses.head()

In [ ]:
n.generators.head()

## Installed capacity by carrier


In [ ]:
cap = n.generators.groupby('carrier').p_nom.sum().sort_values(ascending=False)
cap.round(0)

In [ ]:
import matplotlib.pyplot as plt

ax = cap.plot.bar(figsize=(10, 4))
ax.set_ylabel('Installed capacity (MW)')
ax.set_title(f'{MODEL_NAME}: installed capacity by carrier')
plt.tight_layout(); plt.show()

## Total demand over the year


In [ ]:
demand = n.loads_t.p_set if not n.loads_t.p_set.empty else n.loads_t.p
total = demand.sum(axis=1)
ax = total.plot(figsize=(10, 4))
ax.set_ylabel('Total load (MW)')
ax.set_title(f'{MODEL_NAME}: system demand')
plt.tight_layout(); plt.show()

---
Explore more at [modelverse.bayesian.energy](https://modelverse.bayesian.energy) · powered by [Convexity](https://bayesian.energy).
